In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import os

# --- CONFIGURATION ---
radiomics_file = '../Results/final_radiomics_merged.csv'
output_dir = '../Results/Final_Figures/'
os.makedirs(output_dir, exist_ok=True)

print("--- STARTING XAI DOMAIN AUDIT ---")

# 1. Load Data
if not os.path.exists(radiomics_file):
    print("Error: Merged radiomics file not found.")
else:
    df = pd.read_csv(radiomics_file)
    print(f"Data Loaded: {len(df)} patients")
    
    # 2. Prepare Data for AI
    # Target: 0 = Public, 1 = Local
    df['Target'] = df['Cohort'].apply(lambda x: 1 if 'Local' in x else 0)
    
    # Features: Only radiomics (drop ID, Cohort, Target)
    feature_cols = [c for c in df.columns if 'original_' in c]
    X = df[feature_cols].fillna(0)
    y = df['Target']
    
    # Split Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # 3. Train Classifier
    # We want a model that is GOOD at distinguishing them
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Validation
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)
    print(f"\n--- Model Performance ---")
    print(f"AUC Score: {auc:.4f}")
    print("(AUC > 0.90 confirms distinct domains)")
    
    # 4. SHAP Analysis
    print("\nCalculating SHAP values (This explains the shift)...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # Handle SHAP output format (Binary classification returns a list)
    # We want the values for Class 1 (Local)
    vals_to_plot = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    # 5. Generate Plots
    
    # A. Summary Plot (Beeswarm)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(vals_to_plot, X_test, show=False)
    plt.title("XAI Audit: What makes Bangladeshi Scans Different?", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{output_dir}Figure_4.4_SHAP_Summary.png", dpi=300)
    plt.close()
    print("Saved Figure 4.4 (SHAP Summary)")
    
    # B. Bar Plot (Importance)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(vals_to_plot, X_test, plot_type="bar", show=False)
    plt.title("Top Feature Drivers of Domain Shift", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{output_dir}Figure_4.5_SHAP_Importance.png", dpi=300)
    plt.close()
    print("Saved Figure 4.5 (SHAP Importance)")
    
    print("\nXAI Analysis Complete. You have fulfilled the 'Explainable AI' requirement.")

d:\Anaconda\envs\thesis_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- STARTING XAI DOMAIN AUDIT ---
Data Loaded: 443 patients

--- Model Performance ---
AUC Score: 1.0000
(AUC > 0.90 confirms distinct domains)

Calculating SHAP values (This explains the shift)...
Saved Figure 4.4 (SHAP Summary)
Saved Figure 4.5 (SHAP Importance)

XAI Analysis Complete. You have fulfilled the 'Explainable AI' requirement.


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [13]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import plotly.express as px
import plotly.io as pio
import os

# --- CONFIGURATION ---
radiomics_file = '../Results/final_radiomics_merged.csv' 
output_dir = '../Results/Final_Figures/'
os.makedirs(output_dir, exist_ok=True)

# Set template
pio.templates.default = "plotly_white"

print("--- STARTING MODERN XAI AUDIT (ROBUST FIX) ---")

# 1. Load Data
try:
    df_rad = pd.read_csv(radiomics_file)
    print(f"Data Loaded: {len(df_rad)} patients")
    
    # 2. Prepare Target
    cohorts = df_rad['Cohort'].unique()
    target_map = {c: 1 if 'Local' in c else 0 for c in cohorts}
    df_rad['Target'] = df_rad['Cohort'].map(target_map)
    
    # Get labels
    label_0 = [k for k,v in target_map.items() if v==0][0]
    label_1 = [k for k,v in target_map.items() if v==1][0]
    
    # Features
    feature_cols = [c for c in df_rad.columns if 'original_' in c]
    X = df_rad[feature_cols].fillna(0)
    y = df_rad['Target']
    
    # Clean Names
    clean_names = []
    for col in X.columns:
        name = col.replace('original_', '').replace('wavelet_', '').replace('glcm_', 'Texture: ').replace('shape_', 'Shape: ').replace('firstorder_', 'Stats: ')
        if len(name) > 40: name = name[:37] + "..."
        clean_names.append(name)
    
    X.columns = clean_names
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # 3. Train Model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Validation
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)
    print(f"\n--- AI Performance ---")
    print(f"Task: Distinguish {label_0} vs. {label_1}")
    print(f"AUC Score: {auc:.4f}")
    
    # 4. SHAP Analysis
    print("\nCalculating SHAP...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    # Robustly handle SHAP output (can be list or array)
    if isinstance(shap_values, list):
        # For binary classification, it usually returns a list of [class0_shap, class1_shap]
        # We want Class 1 (Local)
        vals = shap_values[1]
    else:
        # If it returns a single array, use it directly (or check shape)
        if len(shap_values.shape) == 3:
             vals = shap_values[:, :, 1]
        else:
             vals = shap_values
    
    # 5. GENERATE PLOTLY SUMMARY PLOT (MANUAL BEESWARM)
    print("Generating Interactive Plot...")
    
    # Calculate mean absolute SHAP for sorting
    mean_abs_shap = np.abs(vals).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)[-15:] # Top 15 features
    
    # Prepare data for plotting
    plot_data = []
    
    # We loop through the top features
    for i in sorted_idx:
        feature_name = X_test.columns[i]
        
        # Extract columns as 1D arrays
        shap_val = vals[:, i]
        feature_val = X_test.iloc[:, i].values
        
        # Normalize feature value for color mapping (0 to 1)
        # Handle constant features (div by zero)
        denominator = feature_val.max() - feature_val.min()
        if denominator > 0:
            norm_val = (feature_val - feature_val.min()) / denominator
        else:
            norm_val = np.zeros_like(feature_val)
        
        # Add Jitter for Beeswarm effect
        # Create a deterministic jitter based on density or random
        jitter = np.random.normal(0, 0.1, size=len(shap_val)) 
        
        # Construct list of dicts (safe way to build DataFrame)
        for sv, nv, fv, jit in zip(shap_val, norm_val, feature_val, jitter):
            plot_data.append({
                'Feature': feature_name,
                'SHAP Value': float(sv),       # Ensure scalar
                'Normalized Value': float(nv), # Ensure scalar
                'Raw Value': float(fv),        # Ensure scalar
                'Y_Position': i + jit          # Base position (integer) + jitter
            })
            
    df_plot = pd.DataFrame(plot_data)
    
    # Create Scatter Plot (Simulating Beeswarm)
    # Note: We map 'Feature' to hover_name but plot Y using the jittered position
    # We will manually override the Y-axis ticks to show Feature Names
    
    fig = px.scatter(
        df_plot, 
        x='SHAP Value', 
        y='Y_Position', 
        color='Normalized Value',
        color_continuous_scale='RdBu_r', # Red (High) to Blue (Low)
        hover_data=['Feature', 'Raw Value'],
        title=f"<b>XAI Audit: What distinguishes {label_1} from {label_0}?</b><br><sup>Top 15 Features driving the model decision</sup>"
    )
    
    # Customizing Y-Axis to show Feature Names instead of numbers
    # We need the mapping of Index -> Feature Name
    sorted_features = [X_test.columns[i] for i in sorted_idx]
    
    fig.update_layout(
        width=1000,
        height=700,
        font=dict(size=12),
        margin=dict(l=20), 
        coloraxis_colorbar=dict(title="Feature Value<br>(Red=High, Blue=Low)"),
        yaxis=dict(
            title="",
            tickmode='array',
            tickvals=list(range(len(sorted_features))), # 0, 1, 2...
            ticktext=sorted_features # Actual names
        )
    )
    
    # Add vertical line at 0 (Neutral impact)
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
    
    # Save
    fig.write_html(f"{output_dir}Figure_XAI_Interactive.html")
    try:
        fig.write_image(f"{output_dir}Figure_4.4_SHAP_Summary.png", scale=3)
        print("  -> Saved Static PNG")
    except: pass
    
    fig.show()
    
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()

--- STARTING MODERN XAI AUDIT (ROBUST FIX) ---
Data Loaded: 443 patients

--- AI Performance ---
Task: Distinguish Public (Western) vs. Local (Square Hospital)
AUC Score: 1.0000

Calculating SHAP...
Generating Interactive Plot...
  -> Saved Static PNG


In [14]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import plotly.express as px
import plotly.io as pio
import os

# --- CONFIGURATION ---
radiomics_file = '../Results/final_radiomics_merged.csv' 
output_dir = '../Results/Final_Figures/'
os.makedirs(output_dir, exist_ok=True)

# Set template
pio.templates.default = "plotly_white"

print("--- STARTING MODERN XAI AUDIT (FIXED FONTS) ---")

# 1. Load Data
try:
    df_rad = pd.read_csv(radiomics_file)
    
    # 2. Prepare Target
    cohorts = df_rad['Cohort'].unique()
    target_map = {c: 1 if 'Local' in c else 0 for c in cohorts}
    df_rad['Target'] = df_rad['Cohort'].map(target_map)
    
    label_0 = [k for k,v in target_map.items() if v==0][0]
    label_1 = [k for k,v in target_map.items() if v==1][0]
    
    # Features
    feature_cols = [c for c in df_rad.columns if 'original_' in c]
    X = df_rad[feature_cols].fillna(0)
    y = df_rad['Target']
    
    # Clean Names
    clean_names = []
    for col in X.columns:
        name = col.replace('original_', '').replace('wavelet_', '').replace('glcm_', 'Texture: ').replace('shape_', 'Shape: ').replace('firstorder_', 'Stats: ')
        if len(name) > 45: name = name[:42] + "..." # Truncate very long names
        clean_names.append(name)
    
    X.columns = clean_names
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # 3. Train Model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Validation
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)
    print(f"AUC Score: {auc:.4f}")
    
    # 4. SHAP Analysis
    print("Calculating SHAP...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    if isinstance(shap_values, list):
        vals = shap_values[1]
    else:
        if len(shap_values.shape) == 3: vals = shap_values[:, :, 1]
        else: vals = shap_values
    
    # 5. GENERATE PLOTLY SUMMARY PLOT
    print("Generating Interactive Plot...")
    
    mean_abs_shap = np.abs(vals).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)[-15:] # Top 15 features
    
    plot_data = []
    for i in sorted_idx:
        feature_name = X_test.columns[i]
        shap_val = vals[:, i]
        feature_val = X_test.iloc[:, i].values
        
        denominator = feature_val.max() - feature_val.min()
        if denominator > 0:
            norm_val = (feature_val - feature_val.min()) / denominator
        else:
            norm_val = np.zeros_like(feature_val)
        
        jitter = np.random.normal(0, 0.1, size=len(shap_val)) 
        
        for sv, nv, fv, jit in zip(shap_val, norm_val, feature_val, jitter):
            plot_data.append({
                'Feature': feature_name,
                'SHAP Value': float(sv),
                'Normalized Value': float(nv),
                'Raw Value': float(fv),
                'Y_Position': i + jit 
            })
            
    df_plot = pd.DataFrame(plot_data)
    
    fig = px.scatter(
        df_plot, 
        x='SHAP Value', 
        y='Y_Position', 
        color='Normalized Value',
        color_continuous_scale='RdBu_r', 
        hover_data=['Feature', 'Raw Value'],
        title=f"<b>XAI Audit: Drivers of Domain Shift</b><br><sup>What distinguishes {label_1} from {label_0}?</sup>"
    )
    
    sorted_features = [X_test.columns[i] for i in sorted_idx]
    
    # --- LAYOUT FIXES FOR READABILITY ---
    fig.update_layout(
        width=1000,
        height=800, # Increased height to space out labels
        margin=dict(l=300, t=100), # Huge left margin for long text
        font=dict(size=12),
        coloraxis_colorbar=dict(title="Feature Value<br>(Red=High, Blue=Low)"),
        yaxis=dict(
            title="",
            tickmode='array',
            tickvals=list(range(len(sorted_features))),
            ticktext=sorted_features,
            tickfont=dict(size=10) # SMALLER FONT for Y-axis labels
        )
    )
    
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
    
    # Save
    fig.write_html(f"{output_dir}Figure_XAI_Interactive.html")
    try:
        fig.write_image(f"{output_dir}Figure_4.4_SHAP_Summary.png", scale=3)
        print("  -> Saved Static PNG")
    except: pass
    
    fig.show()
    
except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()

--- STARTING MODERN XAI AUDIT (FIXED FONTS) ---
AUC Score: 1.0000
Calculating SHAP...
Generating Interactive Plot...
  -> Saved Static PNG


In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os

# --- CONFIGURATION ---
radiomics_file = '../Results/final_radiomics_merged.csv' 
output_dir = '../Results/Final_Figures/'
os.makedirs(output_dir, exist_ok=True)

# Set template
pio.templates.default = "plotly_white"

print("--- STARTING MODERN XAI AUDIT (FIXED FONTS + WATERFALL) ---")

# 1. Load Data
try:
    df_rad = pd.read_csv(radiomics_file)
    
    # 2. Prepare Target
    cohorts = df_rad['Cohort'].unique()
    target_map = {c: 1 if 'Local' in c else 0 for c in cohorts}
    df_rad['Target'] = df_rad['Cohort'].map(target_map)
    
    label_0 = [k for k,v in target_map.items() if v==0][0]
    label_1 = [k for k,v in target_map.items() if v==1][0]
    
    # Features
    feature_cols = [c for c in df_rad.columns if 'original_' in c]
    X = df_rad[feature_cols].fillna(0)
    y = df_rad['Target']
    
    # Clean Names
    clean_names = []
    for col in X.columns:
        name = col.replace('original_', '').replace('wavelet_', '').replace('glcm_', 'Texture: ').replace('shape_', 'Shape: ').replace('firstorder_', 'Stats: ')
        if len(name) > 45: name = name[:42] + "..." 
        clean_names.append(name)
    
    X.columns = clean_names
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # 3. Train Model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Validation
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)
    print(f"AUC Score: {auc:.4f}")
    
    # 4. SHAP Analysis
    print("Calculating SHAP...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    if isinstance(shap_values, list):
        vals = shap_values[1]
        base_val = explainer.expected_value[1]
    else:
        if len(shap_values.shape) == 3: vals = shap_values[:, :, 1]
        else: vals = shap_values
        base_val = explainer.expected_value

    # --- 5. BEESWARM SUMMARY PLOT ---
    print("Generating Beeswarm Plot...")
    
    mean_abs_shap = np.abs(vals).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)[-15:] 
    
    plot_data = []
    for i in sorted_idx:
        feature_name = X_test.columns[i]
        shap_val = vals[:, i]
        feature_val = X_test.iloc[:, i].values
        
        denominator = feature_val.max() - feature_val.min()
        if denominator > 0:
            norm_val = (feature_val - feature_val.min()) / denominator
        else:
            norm_val = np.zeros_like(feature_val)
        
        jitter = np.random.normal(0, 0.1, size=len(shap_val)) 
        
        for sv, nv, fv, jit in zip(shap_val, norm_val, feature_val, jitter):
            plot_data.append({
                'Feature': feature_name,
                'SHAP Value': float(sv),
                'Normalized Value': float(nv),
                'Raw Value': float(fv),
                'Y_Position': i + jit 
            })
            
    df_plot = pd.DataFrame(plot_data)
    
    fig = px.scatter(
        df_plot, 
        x='SHAP Value', 
        y='Y_Position', 
        color='Normalized Value',
        color_continuous_scale='RdBu_r', 
        hover_data=['Feature', 'Raw Value'],
        title=f"<b>XAI Audit: Population View</b><br><sup>What distinguishes {label_1} from {label_0}?</sup>"
    )
    
    sorted_features = [X_test.columns[i] for i in sorted_idx]
    
    fig.update_layout(
        width=1000,
        height=800,
        margin=dict(l=300, t=100), 
        font=dict(size=12),
        coloraxis_colorbar=dict(title="Feature Value<br>(Red=High, Blue=Low)"),
        yaxis=dict(
            title="",
            tickmode='array',
            tickvals=list(range(len(sorted_features))),
            ticktext=sorted_features,
            tickfont=dict(size=10) 
        )
    )
    
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
    
    fig.write_html(f"{output_dir}Figure_XAI_Beeswarm.html")
    try:
        fig.write_image(f"{output_dir}Figure_4.4_SHAP_Summary.png", scale=3)
        print("  -> Saved Beeswarm PNG")
    except: pass
    
    # --- 6. WATERFALL PLOT (Individual Patient) ---
    print("Generating Waterfall Plot...")
    
    # Select first patient in test set
    pat_idx = 0
    patient_shap = vals[pat_idx]
    patient_data = X_test.iloc[pat_idx]
    
    # Get top 10 contributing features for this patient
    # Sort by absolute SHAP impact
    impact_order = np.argsort(np.abs(patient_shap))[-10:]
    
    wf_names = X_test.columns[impact_order]
    wf_values = patient_shap[impact_order]
    wf_data_values = patient_data.iloc[impact_order]
    
    # Create Waterfall
    fig_wf = go.Figure(go.Waterfall(
        name = "SHAP",
        orientation = "h",
        measure = ["relative"] * len(wf_names),
        y = wf_names,
        x = wf_values,
        connector = {"mode":"between", "line":{"width":4, "color":"rgb(0, 0, 0)", "dash":"solid"}},
        decreasing = {"marker":{"color":"#3498db"}}, # Blue for negative
        increasing = {"marker":{"color":"#e74c3c"}}, # Red for positive
        text = [f"{v:.2f}" for v in wf_values],
        textposition = "outside"
    ))

    fig_wf.update_layout(
        title = f"<b>XAI Audit: Patient Case Study</b><br><sup>Why was this patient classified as {label_1 if y_pred[pat_idx]==1 else label_0}?</sup>",
        showlegend = False,
        width=900,
        height=600,
        margin=dict(l=250),
        xaxis=dict(title="SHAP Value (Contribution to Score)"),
        yaxis=dict(title="", tickfont=dict(size=10))
    )
    
    fig_wf.write_html(f"{output_dir}Figure_XAI_Waterfall.html")
    try:
        fig_wf.write_image(f"{output_dir}Figure_4.5_SHAP_Waterfall.png", scale=3)
        print("  -> Saved Waterfall PNG")
    except: pass
    
    # fig.show() # Commented out to prevent double plot in some notebooks
    # fig_wf.show()

except Exception as e:
    import traceback
    print(f"Error: {e}")
    traceback.print_exc()